In [27]:
import pandas as pd
import ast
import numpy as np
import json

In [28]:
def process_timestamp(dataframe, cutoffdate):
    """
    dataframe
    cutoffdate (pandas datetime): date you want to start processing data from
    """
    def ensure_form(t):
        if t is np.nan: return pd.NaT
        else: return pd.to_datetime(t.get('$date', pd.NaT))

    assert 'timestamp' in dataframe.columns, "Timestamp not in dataframe"
    dataframe['timestamp'] = dataframe['timestamp'].apply(ensure_form, 1)
    timeMask = dataframe.apply(lambda x: x.timestamp >= cutoffdate, 1)

    return dataframe[timeMask]

In [29]:
def ensure_dict(val):
    if isinstance(val, dict):
        return val
    elif isinstance(val, str):
        try:
            return json.loads(val)
        except json.JSONDecodeError:
            return {}
    else:
        return {}

In [30]:
def get_baseline_ih(dfRow):
    # -1 indicates a higher answer is IA, 1 indicates a higher answer is IH 
    survey_mapping =  {
    'When I am really confident in a belief, there is very little chance that belief is wrong.': -1,
    'I\'d rather rely on my own knowledge about most topics than turn to others for expertise.': -1,
    'I am open to revising my important beliefs in the face of new information.': 1,
    'I welcome different ways of thinking about important topics.': 1,
    'I am willing to hear others out, even if I disagree with them in important ways.': 1,
    'I can respect others, even if I disagree with them in important ways.': 1,
    'I can have great respect for someone, even when we don\'t see eye-to-eye on important topics.': 1,
    'Even when I disagree with others, I can recognize that they have sound points.':1
    }
    score, count = 0, 0
    answer_dict = dfRow['IHResponses']
    try:
        for question, answer in answer_dict.items():
            value = int(answer)
            sign = survey_mapping[question]
            if sign == 1:
                score += value 
            if sign == -1:
                score += (11 - value) 
            count += 1
    except AttributeError:
        return None

    if count == 0:
        return None

    return score / count

In [31]:
# Reformat this a bit so we have columns mapping participants to which posts they saw
def clean_posts_column(raw_column):
    if type(raw_column ) is float: return ["", ""]
    post_ids = []
    for i in raw_column:
        post_ids.append(i['$oid'])
    return post_ids

In [32]:
file_path = 'usersround2'
save_path = f"{file_path}/processed_data/"
start_date_naive = pd.to_datetime('06-30-2025')
start_date = start_date_naive.tz_localize('US/Eastern')
treatment_to_treated = True
# Open all the files
comments_df_all = pd.read_json(f"{file_path}/comments.json", lines=True)
participants_df_all = pd.read_json(f"{file_path}/participants.json", lines=True)
survey_df_all = pd.read_json(f"{file_path}/surveys.json", lines=True)
post_df_all = pd.read_json(f"posts.json", lines=True)

In [33]:
# Data Processing for all the files, only get recent data
commentsDf = process_timestamp(comments_df_all, start_date)
participantsDf = process_timestamp(participants_df_all, start_date)

"""
Process the survey data, remove participants that did not pass attention check
Remove participants that do not have a pre and post IH score
"""
survey_df_all.loc[:, 'qna'] = survey_df_all['qna'].apply(ensure_dict)
survey_df_all.loc[:, 'attentionChecker'] = survey_df_all['qna'].apply(lambda d: d.get('attentionChecker'))
survey_df_all.loc[:,'IHResponses'] = survey_df_all['qna'].apply(lambda d: d.get('intellectualHumilityResponses'))
survey_df_all.dropna(subset=['pre_flag'], inplace=True) # quirk from early testing
survey_df_all.loc[:, 'pre_flag'] = survey_df_all['pre_flag'].apply(lambda x: int(x))
surveyDf = survey_df_all[survey_df_all['attentionChecker'] != False].copy() # post IH scores wont have attentionChecker, so they will be None, keep those
passedAttentionCheck = set(surveyDf[surveyDf.attentionChecker == True]['user_id'])
print("Users passed attention check: ", len(passedAttentionCheck))


/var/folders/04/dgz4d5cn0v732mbzpk64d_rc0000gn/T/ipykernel_99428/2429779041.py:11: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  dataframe['timestamp'] = dataframe['timestamp'].apply(ensure_form, 1)


Users passed attention check:  932


/var/folders/04/dgz4d5cn0v732mbzpk64d_rc0000gn/T/ipykernel_99428/2429779041.py:11: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  dataframe['timestamp'] = dataframe['timestamp'].apply(ensure_form, 1)


In [34]:
surveyDf['ih_score'] = surveyDf.apply(get_baseline_ih, 1)
surveyDf['pre_flag'] = surveyDf['pre_flag'].map({1: 'pre_IH', 0: 'post_IH'})
surveyDfSub = surveyDf[['user_id', 'pre_flag', 'ih_score']].copy()
surveyDfSub.dropna(subset=['ih_score'], inplace=True)
surveyDfSub.drop_duplicates(subset=['user_id', 'pre_flag'], inplace=True) # quirk from early testing
pivotedSurveyDf = surveyDfSub.pivot(index='user_id', columns='pre_flag', values='ih_score').reset_index()
pivotedSurveyDf.dropna(inplace=True)
print("Users That Passed Attention and Completed Pre/Post: " , len(pivotedSurveyDf))

Users That Passed Attention and Completed Pre/Post:  737


In [36]:
# Clean up posts
topic_mapping = {
    'Abortion and Reproductive Rights': 0,
    'Action for Climate Change': 1,
    'Undocumented Immigrant Rights and Immigration Policy': 2
}

stance_mapping = {'FOR': 0, 'AGAINST': 1}

post_df_all['clean_post_id'] = post_df_all.apply(lambda x: ast.literal_eval(str(x._id))['$oid'], 1)

In [38]:
print("Starting Eligible: ", len(participantsDf))
participantsDf = participantsDf[participantsDf.completed] 
print("Completed: ", len(participantsDf))
participantMask = participantsDf['user_id'].isin(passedAttentionCheck)
participantsDf = participantsDf[participantMask]
print("Passed Attention Check: ", len(participantsDf))
if treatment_to_treated:
    # OPTION: treatment-to-treated analysis, only those that were actually treated or in non-treatment group
    user_who_were_flagged = set(commentsDf[commentsDf.posted == False].user_id)
    participantsDf['actually_treated'] = participantsDf.apply(lambda x: x.user_id in (list(user_who_were_flagged)), 1)

eligibleParticpants = set(participantsDf['user_id'])

Starting Eligible:  490
Completed:  409
Passed Attention Check:  405


In [39]:
# Combine Posts and Participants Dataframe
participantsDf['clean_posts'] = participantsDf.apply(lambda x: clean_posts_column(x.posts), 1)
participantsDf[['post1', 'post2']] = pd.DataFrame(participantsDf['clean_posts'].tolist(), index=participantsDf.index)

"""
Map participant fields (environment and internvention) to formats good for data analysis.
"""
participantsDf.rename(columns={'intervention':'social_cue_nudge', 'ia_environment':'environment_condtion'}, inplace=True)
participantsDf_w_score = participantsDf.merge(pivotedSurveyDf)
participantsDf_w_score.to_csv(f"{save_path}/participants.csv", index=False)

In [44]:
sorted(list(participantsDf_w_score['post_IH']))

[2.25,
 2.625,
 2.75,
 3.0,
 3.125,
 3.25,
 3.25,
 3.25,
 3.375,
 3.375,
 3.375,
 3.625,
 3.625,
 4.125,
 4.25,
 4.25,
 4.25,
 4.25,
 4.375,
 4.375,
 4.75,
 4.75,
 4.875,
 4.875,
 5.0,
 5.0,
 5.0,
 5.125,
 5.125,
 5.25,
 5.25,
 5.25,
 5.375,
 5.375,
 5.375,
 5.375,
 5.375,
 5.5,
 5.5,
 5.5,
 5.625,
 5.625,
 5.625,
 5.625,
 5.625,
 5.625,
 5.625,
 5.625,
 5.75,
 5.75,
 5.75,
 5.75,
 5.75,
 5.75,
 5.75,
 5.75,
 5.75,
 5.875,
 5.875,
 5.875,
 5.875,
 5.875,
 5.875,
 5.875,
 6.0,
 6.0,
 6.0,
 6.0,
 6.0,
 6.0,
 6.0,
 6.125,
 6.125,
 6.125,
 6.125,
 6.125,
 6.125,
 6.125,
 6.125,
 6.125,
 6.125,
 6.125,
 6.25,
 6.25,
 6.25,
 6.25,
 6.25,
 6.25,
 6.25,
 6.25,
 6.25,
 6.25,
 6.25,
 6.25,
 6.25,
 6.375,
 6.375,
 6.375,
 6.375,
 6.375,
 6.375,
 6.375,
 6.5,
 6.5,
 6.5,
 6.5,
 6.5,
 6.5,
 6.5,
 6.5,
 6.5,
 6.5,
 6.625,
 6.625,
 6.625,
 6.625,
 6.625,
 6.625,
 6.625,
 6.625,
 6.625,
 6.625,
 6.625,
 6.625,
 6.625,
 6.625,
 6.75,
 6.75,
 6.75,
 6.75,
 6.75,
 6.75,
 6.75,
 6.75,
 6.75,
 6.75,
 6.75,

In [26]:
len(participantsDf)

405